In [1]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [2]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars = list(value_dict.values())
    expectation_value = estimator.run(ansatz, qubit_op, pars).result().values
    return np.real(expectation_value[0])  # Ensure it returns a scalar

In [ ]:
min_qubits = 5
max_qubits=6
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar, ASO
    

    problem_dict = {
        "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
        "minmax": "min",
        "obj_func": evaluate_expectation
    }

    model = ASO.OriginalASO(epoch=1000, pop_size=100, alpha = 50, beta = 0.2)
    g_best = model.solve(problem_dict)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 04:55:13 PM, INFO, mealpy.physics_based.ASO.OriginalASO: Solving single objective optimization problem.


ansatz_num_parameters= 20


2024/07/10 04:55:14 PM, INFO, mealpy.physics_based.ASO.OriginalASO: >>>Problem: P, Epoch: 1, Current best: -2.15625, Global best: -2.15625, Runtime: 0.46595 seconds
2024/07/10 04:55:15 PM, INFO, mealpy.physics_based.ASO.OriginalASO: >>>Problem: P, Epoch: 2, Current best: -2.15625, Global best: -2.15625, Runtime: 0.47592 seconds
2024/07/10 04:55:15 PM, INFO, mealpy.physics_based.ASO.OriginalASO: >>>Problem: P, Epoch: 3, Current best: -2.15625, Global best: -2.15625, Runtime: 0.48877 seconds
2024/07/10 04:55:16 PM, INFO, mealpy.physics_based.ASO.OriginalASO: >>>Problem: P, Epoch: 4, Current best: -2.15625, Global best: -2.15625, Runtime: 0.49518 seconds
2024/07/10 04:55:16 PM, INFO, mealpy.physics_based.ASO.OriginalASO: >>>Problem: P, Epoch: 5, Current best: -2.15625, Global best: -2.15625, Runtime: 0.45204 seconds
2024/07/10 04:55:17 PM, INFO, mealpy.physics_based.ASO.OriginalASO: >>>Problem: P, Epoch: 6, Current best: -2.15625, Global best: -2.15625, Runtime: 0.44334 seconds
2024/07/10

KeyboardInterrupt: 

In [6]:
from mealpy import FloatVar, ArchOA
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = ArchOA.OriginalArchOA(epoch=1000, pop_size=50, c1 = 2, c2 = 5, c3 = 2, c4 = 0.5, acc_max = 0.9, acc_min = 0.1)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")


2024/07/10 05:01:01 PM, INFO, mealpy.physics_based.ArchOA.OriginalArchOA: Solving single objective optimization problem.
2024/07/10 05:01:01 PM, INFO, mealpy.physics_based.ArchOA.OriginalArchOA: >>>Problem: P, Epoch: 1, Current best: -2.6875, Global best: -2.6875, Runtime: 0.20877 seconds
2024/07/10 05:01:02 PM, INFO, mealpy.physics_based.ArchOA.OriginalArchOA: >>>Problem: P, Epoch: 2, Current best: -2.6875, Global best: -2.6875, Runtime: 0.16593 seconds
2024/07/10 05:01:02 PM, INFO, mealpy.physics_based.ArchOA.OriginalArchOA: >>>Problem: P, Epoch: 3, Current best: -2.6875, Global best: -2.6875, Runtime: 0.17858 seconds
2024/07/10 05:01:02 PM, INFO, mealpy.physics_based.ArchOA.OriginalArchOA: >>>Problem: P, Epoch: 4, Current best: -2.6875, Global best: -2.6875, Runtime: 0.18360 seconds
2024/07/10 05:01:02 PM, INFO, mealpy.physics_based.ArchOA.OriginalArchOA: >>>Problem: P, Epoch: 5, Current best: -2.6875, Global best: -2.6875, Runtime: 0.16912 seconds
2024/07/10 05:01:02 PM, INFO, meal

KeyboardInterrupt: 

In [8]:
import numpy as np
from mealpy import FloatVar, CDO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = CDO.OriginalCDO(epoch=1000, pop_size=100)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")


2024/07/10 05:06:32 PM, INFO, mealpy.physics_based.CDO.OriginalCDO: Solving single objective optimization problem.


2024/07/10 05:06:33 PM, INFO, mealpy.physics_based.CDO.OriginalCDO: >>>Problem: P, Epoch: 1, Current best: -1.21875, Global best: -2.09375, Runtime: 0.37371 seconds
2024/07/10 05:06:33 PM, INFO, mealpy.physics_based.CDO.OriginalCDO: >>>Problem: P, Epoch: 2, Current best: -1.09375, Global best: -2.09375, Runtime: 0.39893 seconds
2024/07/10 05:06:33 PM, INFO, mealpy.physics_based.CDO.OriginalCDO: >>>Problem: P, Epoch: 3, Current best: -1.0, Global best: -2.09375, Runtime: 0.37785 seconds
2024/07/10 05:06:34 PM, INFO, mealpy.physics_based.CDO.OriginalCDO: >>>Problem: P, Epoch: 4, Current best: -1.4375, Global best: -2.09375, Runtime: 0.47297 seconds
2024/07/10 05:06:34 PM, INFO, mealpy.physics_based.CDO.OriginalCDO: >>>Problem: P, Epoch: 5, Current best: -1.375, Global best: -2.09375, Runtime: 0.36766 seconds
2024/07/10 05:06:35 PM, INFO, mealpy.physics_based.CDO.OriginalCDO: >>>Problem: P, Epoch: 6, Current best: -1.53125, Global best: -2.09375, Runtime: 0.32828 seconds
2024/07/10 05:06:

KeyboardInterrupt: 

In [11]:
from mealpy import FloatVar, EFO
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = EFO.DevEFO(epoch=1000, pop_size=50, r_rate = 0.3, ps_rate = 0.85, p_field = 0.1, n_field = 0.45)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 05:13:03 PM, INFO, mealpy.physics_based.EFO.DevEFO: Solving single objective optimization problem.
2024/07/10 05:13:03 PM, INFO, mealpy.physics_based.EFO.DevEFO: >>>Problem: P, Epoch: 1, Current best: -1.59375, Global best: -1.59375, Runtime: 0.15508 seconds
2024/07/10 05:13:04 PM, INFO, mealpy.physics_based.EFO.DevEFO: >>>Problem: P, Epoch: 2, Current best: -1.59375, Global best: -1.59375, Runtime: 0.15409 seconds
2024/07/10 05:13:04 PM, INFO, mealpy.physics_based.EFO.DevEFO: >>>Problem: P, Epoch: 3, Current best: -1.59375, Global best: -1.59375, Runtime: 0.16055 seconds
2024/07/10 05:13:04 PM, INFO, mealpy.physics_based.EFO.DevEFO: >>>Problem: P, Epoch: 4, Current best: -1.59375, Global best: -1.59375, Runtime: 0.15432 seconds
2024/07/10 05:13:04 PM, INFO, mealpy.physics_based.EFO.DevEFO: >>>Problem: P, Epoch: 5, Current best: -1.59375, Global best: -1.59375, Runtime: 0.16007 seconds
2024/07/10 05:13:04 PM, INFO, mealpy.physics_based.EFO.DevEFO: >>>Problem: P, Epoch: 6, Cu

KeyboardInterrupt: 

In [14]:
from mealpy import FloatVar, EO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = EO.OriginalEO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 05:21:22 PM, INFO, mealpy.physics_based.EO.OriginalEO: Solving single objective optimization problem.


2024/07/10 05:21:22 PM, INFO, mealpy.physics_based.EO.OriginalEO: >>>Problem: P, Epoch: 1, Current best: -1.9375, Global best: -1.9375, Runtime: 0.09932 seconds
2024/07/10 05:21:22 PM, INFO, mealpy.physics_based.EO.OriginalEO: >>>Problem: P, Epoch: 2, Current best: -1.9375, Global best: -1.9375, Runtime: 0.09479 seconds
2024/07/10 05:21:22 PM, INFO, mealpy.physics_based.EO.OriginalEO: >>>Problem: P, Epoch: 3, Current best: -1.9375, Global best: -1.9375, Runtime: 0.07970 seconds
2024/07/10 05:21:22 PM, INFO, mealpy.physics_based.EO.OriginalEO: >>>Problem: P, Epoch: 4, Current best: -1.9375, Global best: -1.9375, Runtime: 0.07800 seconds
2024/07/10 05:21:22 PM, INFO, mealpy.physics_based.EO.OriginalEO: >>>Problem: P, Epoch: 5, Current best: -1.9375, Global best: -1.9375, Runtime: 0.09298 seconds
2024/07/10 05:21:22 PM, INFO, mealpy.physics_based.EO.OriginalEO: >>>Problem: P, Epoch: 6, Current best: -1.9375, Global best: -1.9375, Runtime: 0.09720 seconds
2024/07/10 05:21:23 PM, INFO, meal

Solution: [  8.63705869  11.9318961   90.70853482 -73.72811026 -43.96956298
  65.38834692 -66.5607628  -41.62562366 -85.99872874  32.31121448
  26.92935771  14.05414689 -80.38673354  36.61477622 -19.7179976
 -89.43414747  -4.65034363 -32.68929603  46.14976713 -58.71523269], Fitness: -3.84375
Solution: [  8.63705869  11.9318961   90.70853482 -73.72811026 -43.96956298
  65.38834692 -66.5607628  -41.62562366 -85.99872874  32.31121448
  26.92935771  14.05414689 -80.38673354  36.61477622 -19.7179976
 -89.43414747  -4.65034363 -32.68929603  46.14976713 -58.71523269], Fitness: -3.84375


In [15]:
from mealpy import FloatVar, EVO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = EVO.OriginalEVO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 05:23:00 PM, INFO, mealpy.physics_based.EVO.OriginalEVO: Solving single objective optimization problem.
2024/07/10 05:23:01 PM, INFO, mealpy.physics_based.EVO.OriginalEVO: >>>Problem: P, Epoch: 1, Current best: -1.28125, Global best: -1.28125, Runtime: 0.25252 seconds
2024/07/10 05:23:01 PM, INFO, mealpy.physics_based.EVO.OriginalEVO: >>>Problem: P, Epoch: 2, Current best: -2.1875, Global best: -2.1875, Runtime: 0.29621 seconds
2024/07/10 05:23:01 PM, INFO, mealpy.physics_based.EVO.OriginalEVO: >>>Problem: P, Epoch: 3, Current best: -2.1875, Global best: -2.1875, Runtime: 0.29337 seconds
2024/07/10 05:23:01 PM, INFO, mealpy.physics_based.EVO.OriginalEVO: >>>Problem: P, Epoch: 4, Current best: -2.1875, Global best: -2.1875, Runtime: 0.27829 seconds
2024/07/10 05:23:02 PM, INFO, mealpy.physics_based.EVO.OriginalEVO: >>>Problem: P, Epoch: 5, Current best: -2.1875, Global best: -2.1875, Runtime: 0.26569 seconds
2024/07/10 05:23:02 PM, INFO, mealpy.physics_based.EVO.OriginalEVO: 

Solution: [ -1.07980549 -30.04506233  28.55003026 -31.18210153  10.21505859
  90.57990351  -4.98763435 -80.31434804 -54.02459929  85.91345485
  35.73730367 -61.79961733 -39.19937739  99.19879282 -73.9187093
  30.17468416 -71.53000179 -73.37437437 -79.99742192  40.74625349], Fitness: -2.40625
Solution: [ -1.07980549 -30.04506233  28.55003026 -31.18210153  10.21505859
  90.57990351  -4.98763435 -80.31434804 -54.02459929  85.91345485
  35.73730367 -61.79961733 -39.19937739  99.19879282 -73.9187093
  30.17468416 -71.53000179 -73.37437437 -79.99742192  40.74625349], Fitness: -2.40625


In [18]:
from mealpy import FloatVar, FLA
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = FLA.OriginalFLA(epoch=1000, pop_size=50, C1 = 0.5, C2 = 2.0, C3 = 0.1, C4 = 0.2, C5 = 2.0, DD = 0.01)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 05:30:17 PM, INFO, mealpy.physics_based.FLA.OriginalFLA: Solving single objective optimization problem.
2024/07/10 05:30:17 PM, INFO, mealpy.physics_based.FLA.OriginalFLA: >>>Problem: P, Epoch: 1, Current best: -1.625, Global best: -1.625, Runtime: 0.19595 seconds
2024/07/10 05:30:18 PM, INFO, mealpy.physics_based.FLA.OriginalFLA: >>>Problem: P, Epoch: 2, Current best: -2.34375, Global best: -2.34375, Runtime: 0.20149 seconds
2024/07/10 05:30:18 PM, INFO, mealpy.physics_based.FLA.OriginalFLA: >>>Problem: P, Epoch: 3, Current best: -2.40625, Global best: -2.40625, Runtime: 0.18694 seconds
2024/07/10 05:30:18 PM, INFO, mealpy.physics_based.FLA.OriginalFLA: >>>Problem: P, Epoch: 4, Current best: -2.40625, Global best: -2.40625, Runtime: 0.18192 seconds
2024/07/10 05:30:18 PM, INFO, mealpy.physics_based.FLA.OriginalFLA: >>>Problem: P, Epoch: 5, Current best: -2.40625, Global best: -2.40625, Runtime: 0.20128 seconds
2024/07/10 05:30:18 PM, INFO, mealpy.physics_based.FLA.OriginalF

KeyboardInterrupt: 

In [20]:
from mealpy import FloatVar, HGSO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = HGSO.OriginalHGSO(epoch=1000, pop_size=50, n_clusters = 3)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 05:33:42 PM, INFO, mealpy.physics_based.HGSO.OriginalHGSO: Solving single objective optimization problem.
2024/07/10 05:33:42 PM, INFO, mealpy.physics_based.HGSO.OriginalHGSO: >>>Problem: P, Epoch: 1, Current best: -1.25, Global best: -1.6875, Runtime: 0.18551 seconds
2024/07/10 05:33:42 PM, INFO, mealpy.physics_based.HGSO.OriginalHGSO: >>>Problem: P, Epoch: 2, Current best: -0.84375, Global best: -1.6875, Runtime: 0.18944 seconds
2024/07/10 05:33:43 PM, INFO, mealpy.physics_based.HGSO.OriginalHGSO: >>>Problem: P, Epoch: 3, Current best: -1.21875, Global best: -1.6875, Runtime: 0.18664 seconds
2024/07/10 05:33:43 PM, INFO, mealpy.physics_based.HGSO.OriginalHGSO: >>>Problem: P, Epoch: 4, Current best: -0.875, Global best: -1.6875, Runtime: 0.22474 seconds
2024/07/10 05:33:43 PM, INFO, mealpy.physics_based.HGSO.OriginalHGSO: >>>Problem: P, Epoch: 5, Current best: -1.03125, Global best: -1.6875, Runtime: 0.18857 seconds
2024/07/10 05:33:43 PM, INFO, mealpy.physics_based.HGSO.Or

KeyboardInterrupt: 

In [23]:
from mealpy import FloatVar, MVO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = MVO.DevMVO(epoch=1000, pop_size=50, wep_min = 0.2, wep_max = 1.0)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 05:39:50 PM, INFO, mealpy.physics_based.MVO.DevMVO: Solving single objective optimization problem.
2024/07/10 05:39:51 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 1, Current best: -1.59375, Global best: -1.59375, Runtime: 0.19637 seconds
2024/07/10 05:39:51 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 2, Current best: -1.78125, Global best: -1.78125, Runtime: 0.18162 seconds
2024/07/10 05:39:51 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 3, Current best: -1.78125, Global best: -1.78125, Runtime: 0.17641 seconds
2024/07/10 05:39:51 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 4, Current best: -1.78125, Global best: -1.78125, Runtime: 0.20939 seconds
2024/07/10 05:39:51 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 5, Current best: -1.78125, Global best: -1.78125, Runtime: 0.20468 seconds
2024/07/10 05:39:52 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 6, Cu

Solution: [  12.09925784   49.1732633    90.48072489   61.05325818   87.58251942
   65.0397461    99.14107131  -95.9777163    64.4081246   -56.67799682
   -1.30451468   36.04844794   51.72370639  -10.93711176   82.81640936
  -32.5650969    -8.19144741   -1.38958161 -100.           39.2475608 ], Fitness: -3.9375
Solution: [  12.09925784   49.1732633    90.48072489   61.05325818   87.58251942
   65.0397461    99.14107131  -95.9777163    64.4081246   -56.67799682
   -1.30451468   36.04844794   51.72370639  -10.93711176   82.81640936
  -32.5650969    -8.19144741   -1.38958161 -100.           39.2475608 ], Fitness: -3.9375


In [22]:
from mealpy import FloatVar, MVO


problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = MVO.DevMVO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 05:38:34 PM, INFO, mealpy.physics_based.MVO.DevMVO: Solving single objective optimization problem.
2024/07/10 05:38:35 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 1, Current best: -1.5, Global best: -1.5, Runtime: 0.18778 seconds
2024/07/10 05:38:35 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 2, Current best: -1.5, Global best: -1.5, Runtime: 0.20747 seconds
2024/07/10 05:38:35 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 3, Current best: -1.5, Global best: -1.5, Runtime: 0.23207 seconds
2024/07/10 05:38:36 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 4, Current best: -1.5, Global best: -1.5, Runtime: 0.21484 seconds
2024/07/10 05:38:36 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 5, Current best: -1.625, Global best: -1.625, Runtime: 0.21456 seconds
2024/07/10 05:38:36 PM, INFO, mealpy.physics_based.MVO.DevMVO: >>>Problem: P, Epoch: 6, Current best: -1.625, Global best: -1.

KeyboardInterrupt: 

In [26]:
from mealpy import FloatVar, RIME


problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = RIME.OriginalRIME(epoch=1000, pop_size=50, sr = 5.0)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 07:01:58 PM, INFO, mealpy.physics_based.RIME.OriginalRIME: Solving single objective optimization problem.
2024/07/10 07:01:59 PM, INFO, mealpy.physics_based.RIME.OriginalRIME: >>>Problem: P, Epoch: 1, Current best: -3.625, Global best: -3.625, Runtime: 0.17770 seconds
2024/07/10 07:01:59 PM, INFO, mealpy.physics_based.RIME.OriginalRIME: >>>Problem: P, Epoch: 2, Current best: -3.625, Global best: -3.625, Runtime: 0.18777 seconds
2024/07/10 07:01:59 PM, INFO, mealpy.physics_based.RIME.OriginalRIME: >>>Problem: P, Epoch: 3, Current best: -3.625, Global best: -3.625, Runtime: 0.18106 seconds
2024/07/10 07:01:59 PM, INFO, mealpy.physics_based.RIME.OriginalRIME: >>>Problem: P, Epoch: 4, Current best: -3.625, Global best: -3.625, Runtime: 0.22093 seconds
2024/07/10 07:02:00 PM, INFO, mealpy.physics_based.RIME.OriginalRIME: >>>Problem: P, Epoch: 5, Current best: -3.625, Global best: -3.625, Runtime: 0.24725 seconds
2024/07/10 07:02:00 PM, INFO, mealpy.physics_based.RIME.OriginalRIME

KeyboardInterrupt: 

In [27]:
from mealpy import FloatVar, TWO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = TWO.OriginalTWO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 07:02:56 PM, INFO, mealpy.physics_based.TWO.OriginalTWO: Solving single objective optimization problem.
2024/07/10 07:02:57 PM, INFO, mealpy.physics_based.TWO.OriginalTWO: >>>Problem: P, Epoch: 1, Current best: -1.90625, Global best: -1.90625, Runtime: 0.19653 seconds
2024/07/10 07:02:57 PM, INFO, mealpy.physics_based.TWO.OriginalTWO: >>>Problem: P, Epoch: 2, Current best: -1.90625, Global best: -1.90625, Runtime: 0.19067 seconds
2024/07/10 07:02:57 PM, INFO, mealpy.physics_based.TWO.OriginalTWO: >>>Problem: P, Epoch: 3, Current best: -1.90625, Global best: -1.90625, Runtime: 0.25413 seconds
2024/07/10 07:02:58 PM, INFO, mealpy.physics_based.TWO.OriginalTWO: >>>Problem: P, Epoch: 4, Current best: -1.90625, Global best: -1.90625, Runtime: 0.26030 seconds
2024/07/10 07:02:58 PM, INFO, mealpy.physics_based.TWO.OriginalTWO: >>>Problem: P, Epoch: 5, Current best: -1.90625, Global best: -1.90625, Runtime: 0.20583 seconds
2024/07/10 07:02:58 PM, INFO, mealpy.physics_based.TWO.Origi

KeyboardInterrupt: 

In [28]:
from mealpy import FloatVar, WDO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = WDO.OriginalWDO(epoch=1000, pop_size=50, RT = 3, g_c = 0.2, alp = 0.4, c_e = 0.4, max_v = 0.3)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 07:04:37 PM, INFO, mealpy.physics_based.WDO.OriginalWDO: Solving single objective optimization problem.
2024/07/10 07:04:38 PM, INFO, mealpy.physics_based.WDO.OriginalWDO: >>>Problem: P, Epoch: 1, Current best: -1.28125, Global best: -1.28125, Runtime: 0.19216 seconds
2024/07/10 07:04:38 PM, INFO, mealpy.physics_based.WDO.OriginalWDO: >>>Problem: P, Epoch: 2, Current best: -1.28125, Global best: -1.28125, Runtime: 0.19813 seconds
2024/07/10 07:04:38 PM, INFO, mealpy.physics_based.WDO.OriginalWDO: >>>Problem: P, Epoch: 3, Current best: -1.28125, Global best: -1.28125, Runtime: 0.22912 seconds
2024/07/10 07:04:38 PM, INFO, mealpy.physics_based.WDO.OriginalWDO: >>>Problem: P, Epoch: 4, Current best: -1.28125, Global best: -1.28125, Runtime: 0.20768 seconds
2024/07/10 07:04:38 PM, INFO, mealpy.physics_based.WDO.OriginalWDO: >>>Problem: P, Epoch: 5, Current best: -1.28125, Global best: -1.28125, Runtime: 0.20963 seconds
2024/07/10 07:04:39 PM, INFO, mealpy.physics_based.WDO.Origi

KeyboardInterrupt: 